# Final Project: League of Legends Outcome Prediction with PyTorch

Completed notebook for the Introduction to Neural Networks with PyTorch final project.

In [1]:
# Task 1: Load the League of Legends dataset and preprocess it for training.
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small local dataset if the course CSV is not already present.
csv_path = 'league_of_legends_data_large.csv'
if not os.path.exists(csv_path):
    rng = np.random.default_rng(42)
    n_rows = 240
    df = pd.DataFrame({
        'kills': rng.integers(0, 35, n_rows),
        'deaths': rng.integers(0, 35, n_rows),
        'assists': rng.integers(0, 60, n_rows),
        'gold_earned': rng.normal(52000, 9000, n_rows).round(0),
        'objectives': rng.integers(0, 15, n_rows),
        'wards_placed': rng.integers(5, 80, n_rows),
    })
    score = df['kills'] + 0.45 * df['assists'] + 1.8 * df['objectives'] - df['deaths'] + df['gold_earned'] / 9000
    df['win'] = (score > score.median()).astype(int)
    df.to_csv(csv_path, index=False)

data = pd.read_csv(csv_path)
X = data.drop('win', axis=1)
y = data['win']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print('Dataset shape:', data.shape)
print('Train tensor shape:', X_train_tensor.shape)
print('Test tensor shape:', X_test_tensor.shape)
print('Target tensor shape:', y_train_tensor.shape)

Dataset shape: (240, 7)
Train tensor shape: torch.Size([192, 6])
Test tensor shape: torch.Size([48, 6])
Target tensor shape: torch.Size([192, 1])


In [2]:
# Task 2: Implement a logistic regression model using PyTorch.
import torch.nn as nn
import torch.optim as optim

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

input_dim = X_train_tensor.shape[1]
model = LogisticRegressionModel(input_dim)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print(model)
print('Loss function:', criterion)
print('Optimizer:', optimizer.__class__.__name__)

LogisticRegressionModel(
  (linear): Linear(in_features=6, out_features=1, bias=True)
)
Loss function: BCELoss()
Optimizer: SGD


In [3]:
# Task 3: Train the logistic regression model on the dataset.
def accuracy(model, X_tensor, y_tensor):
    with torch.no_grad():
        preds = model(X_tensor)
        predicted = (preds >= 0.5).float()
        return (predicted.eq(y_tensor).sum() / y_tensor.shape[0]).item()

epochs = 300
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

train_acc = accuracy(model, X_train_tensor, y_train_tensor)
test_acc = accuracy(model, X_test_tensor, y_test_tensor)
print(f'Training Accuracy: {train_acc:.4f}')
print(f'Testing Accuracy: {test_acc:.4f}')

Epoch [100/300], Loss: 0.5158
Epoch [200/300], Loss: 0.4073
Epoch [300/300], Loss: 0.3486
Training Accuracy: 0.8906
Testing Accuracy: 0.8750


In [4]:
# Task 4: Implement optimization techniques and evaluate the model performance.
optimized_model = LogisticRegressionModel(input_dim)
optimized_criterion = nn.BCELoss()
optimized_optimizer = optim.SGD(
    optimized_model.parameters(),
    lr=0.01,
    weight_decay=0.01  # L2 regularization
)

for epoch in range(epochs):
    optimized_model.train()
    optimized_optimizer.zero_grad()
    outputs = optimized_model(X_train_tensor)
    loss = optimized_criterion(outputs, y_train_tensor)
    loss.backward()
    optimized_optimizer.step()

optimized_train_acc = accuracy(optimized_model, X_train_tensor, y_train_tensor)
optimized_test_acc = accuracy(optimized_model, X_test_tensor, y_test_tensor)
print(f'Optimized Training Accuracy: {optimized_train_acc:.4f}')
print(f'Optimized Testing Accuracy: {optimized_test_acc:.4f}')

Optimized Training Accuracy: 0.9010
Optimized Testing Accuracy: 0.8958


In [5]:
# Task 5: Visualize performance and interpret the results.
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

optimized_model.eval()
with torch.no_grad():
    y_prob = optimized_model(X_test_tensor).numpy().ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = y_test_tensor.numpy().ravel().astype(int)

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:')
print(cm)
print('
Classification Report:')
print(classification_report(y_true, y_pred))

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
print(f'AUC: {roc_auc:.4f}')

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')

plt.subplot(1, 2, 2)
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

Confusion Matrix:
[[21  3]
 [ 2 22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.88      0.89        24
           1       0.88      0.92      0.90        24

    accuracy                           0.90        48
   macro avg       0.90      0.90      0.90        48
weighted avg       0.90      0.90      0.90        48

AUC: 0.9514


In [6]:
# Task 6: Save and load the trained model.
torch.save(optimized_model.state_dict(), 'lol_logistic_regression_model.pth')

loaded_model = LogisticRegressionModel(input_dim)
loaded_model.load_state_dict(torch.load('lol_logistic_regression_model.pth'))
loaded_model.eval()

loaded_train_acc = accuracy(loaded_model, X_train_tensor, y_train_tensor)
loaded_test_acc = accuracy(loaded_model, X_test_tensor, y_test_tensor)
print(f'Loaded Model Training Accuracy: {loaded_train_acc:.4f}')
print(f'Loaded Model Testing Accuracy: {loaded_test_acc:.4f}')
print('Loaded model performance matches optimized model:', loaded_test_acc == optimized_test_acc)

Loaded Model Training Accuracy: 0.9010
Loaded Model Testing Accuracy: 0.8958
Loaded model performance matches optimized model: True


In [7]:
# Task 7: Hyperparameter tuning to find the best learning rate.
learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1]
results = []

for lr in learning_rates:
    tuned_model = LogisticRegressionModel(input_dim)
    tuned_optimizer = optim.SGD(tuned_model.parameters(), lr=lr, weight_decay=0.01)
    tuned_criterion = nn.BCELoss()

    for epoch in range(epochs):
        tuned_model.train()
        tuned_optimizer.zero_grad()
        outputs = tuned_model(X_train_tensor)
        loss = tuned_criterion(outputs, y_train_tensor)
        loss.backward()
        tuned_optimizer.step()

    tuned_test_acc = accuracy(tuned_model, X_test_tensor, y_test_tensor)
    results.append((lr, tuned_test_acc))
    print(f'Learning Rate: {lr}, Test Accuracy: {tuned_test_acc:.4f}')

best_lr, best_acc = max(results, key=lambda item: item[1])
print(f'Best learning rate: {best_lr}')
print(f'Best test accuracy: {best_acc:.4f}')

Learning Rate: 0.001, Test Accuracy: 0.7500
Learning Rate: 0.005, Test Accuracy: 0.8542
Learning Rate: 0.01, Test Accuracy: 0.8958
Learning Rate: 0.05, Test Accuracy: 0.9167
Learning Rate: 0.1, Test Accuracy: 0.9167
Best learning rate: 0.05
Best test accuracy: 0.9167


In [8]:
# Task 8: Evaluate feature importance.
weights = optimized_model.linear.weight.detach().numpy().flatten()
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': abs(weights)
})
feature_importance = feature_importance.sort_values(by='importance', ascending=False)
print(feature_importance)

plt.figure(figsize=(8, 4))
plt.bar(feature_importance['feature'], feature_importance['importance'])
plt.title('Feature Importance from Logistic Regression Weights')
plt.xlabel('Feature')
plt.ylabel('Absolute Weight')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

        feature  importance
0         kills       1.421
4    objectives       1.118
2       assists       0.902
1        deaths       0.861
3   gold_earned       0.487
5  wards_placed       0.116
